# Explore LIBERO v2

Sibling of `explore_libero_v1.ipynb` for the v2 HDF5 dataset.

Sections:
1. Open a per-task HDF5 and dump its schema
2. Single-trial inspection (window storyboard, depth strips, contact cloud, goal markers)
3. Sibling-group consistency (3 trials of the same `(demo, bin)` share window arrays)
4. Velocity reality check (v2 `window_qvel` non-zero, vs v1's ≈0)
5. Random-access throughput on v2 reader
6. Label-form round-trip: rebuild v1 agentview heatmap from v2 storage

Default path is the recommended external-drive layout; edit `V2_ROOT` to point elsewhere.

In [ ]:
from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt

from planner.risk.v2_store import V2Reader
from planner.risk.dataset_v2 import LiberoV2Dataset

V2_ROOT = Path('/media/aaron/F/failbench/libero/v2')
SPLIT = 'libero_spatial'

tasks = sorted((V2_ROOT / SPLIT).glob('*.h5'))
print(f'{len(tasks)} task HDF5 files under {V2_ROOT}/{SPLIT}')
for p in tasks[:5]:
    print(' ', p.name)

## 1. Schema dump

In [ ]:
import h5py

path = tasks[0]
with h5py.File(path, 'r') as f:
    print(f'File: {path.name}')
    print(f'File attrs:')
    for k, v in f.attrs.items():
        print(f'  {k}: {v}')
    trial_ids = sorted(f['trials'].keys())
    print(f'\nTrials: {len(trial_ids)} (showing first 5)')
    for tid in trial_ids[:5]:
        print(' ', tid)
    print(f'\nDatasets in /trials/{trial_ids[0]}:')
    g = f[f'trials/{trial_ids[0]}']
    for k, v in g.items():
        print(f'  {k:35s}  shape={tuple(v.shape):30s}  dtype={v.dtype}')
    print(f'\nAttrs on /trials/{trial_ids[0]}:')
    for k, v in g.attrs.items():
        s = str(v)
        if len(s) > 80:
            s = s[:77] + '...'
        print(f'  {k}: {s}')

## 2. Single-trial storyboard

In [ ]:
ds = LiberoV2Dataset(V2_ROOT, splits=(SPLIT,), use_settle=True)
print(f'Dataset size: {len(ds)}')

# Pick a trial that actually has contacts.
for idx in range(len(ds)):
    s = ds[idx]
    if s['contact_positions'].shape[0] > 100:
        break
print(f"Trial {s['trial_id']}: failure={s['failure_mode']} "
      f"joints={list(s['failure_joints'])} is_holding={s['is_holding']} "
      f"contacts={s['contact_positions'].shape[0]}")

In [ ]:
# Window storyboard: agentview + wrist RGB strips, T frames each.
T = s['window_agentview_rgb'].shape[0]
fig, axes = plt.subplots(2, T, figsize=(2.2 * T, 4.4))
for k in range(T):
    axes[0, k].imshow(s['window_agentview_rgb'][k])
    axes[0, k].set_title(f'agent t-{T-1-k}\nidx={s["window_frame_idx"][k]}', fontsize=8)
    axes[0, k].axis('off')
    axes[1, k].imshow(s['window_wrist_rgb'][k])
    axes[1, k].set_title('wrist', fontsize=8)
    axes[1, k].axis('off')
fig.suptitle(f"Pre-failure window — {s['trial_id']}")
plt.tight_layout(); plt.show()

In [ ]:
# pre / post comparison with contact overlay on post.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(s['pre_rgb']); axes[0].set_title('pre_rgb (fail moment)'); axes[0].axis('off')
axes[1].imshow(s['post_agentview_rgb']); axes[1].set_title('post_agentview_rgb (after settle)'); axes[1].axis('off')
axes[2].imshow(s['pre_depth'].astype(np.float32), cmap='viridis', vmin=0.5, vmax=2.0)
axes[2].set_title('pre_depth (m)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 3D contact scatter, coloured by contact_time.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
p = s['contact_positions']
ct = s['contact_time']
sc = ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=ct, cmap='viridis', s=2)
fig.colorbar(sc, ax=ax, label='settle step')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_title(f"World-frame contacts ({len(p)} total), {s['failure_mode']}")
plt.tight_layout(); plt.show()

## 3. Sibling-group consistency

Trials sharing the same `(demo_key, bin_idx)` but different `seed_idx` should have **identical window arrays** (same fail_idx → same window seeding) yet **different contact clouds** (different sampled failures).

In [ ]:
from collections import defaultdict

groups = defaultdict(list)
for i, idx in enumerate(ds._index):
    groups[(idx.task, idx.trial_id.split('_s')[0], idx.trial_id.split('_b')[1])].append(i)
multi = [g for g in groups.values() if len(g) >= 2][:3]

for group in multi:
    samples = [ds[i] for i in group]
    win_match = all(np.allclose(samples[0]['window_qpos'], s['window_qpos'])
                    for s in samples[1:])
    contact_counts = [s['contact_positions'].shape[0] for s in samples]
    modes = [s['failure_mode'] for s in samples]
    print(f"group {samples[0]['trial_id'].split('_s')[0]}_b{group}: window_match={win_match} "
          f"modes={modes} n_contacts={contact_counts}")

## 4. Velocity reality check

In [ ]:
qvel_norms = []
for i in range(min(200, len(ds))):
    s = ds[i]
    qvel_norms.append(float(np.linalg.norm(s['window_qvel'][-1])))

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(qvel_norms, bins=40)
ax.set_xlabel('||window_qvel[-1]||  (rad/s)')
ax.set_ylabel('# trials')
ax.set_title('v2 has real joint velocity at fail moment (v1 was ≈0)')
plt.tight_layout(); plt.show()
print(f'median: {np.median(qvel_norms):.3f} rad/s    max: {max(qvel_norms):.3f}')

## 5. Random-access throughput

In [ ]:
rng = np.random.default_rng(0)
indices = rng.integers(0, len(ds), size=100)

# Warm read (page cache).
for i in indices[:10]:
    _ = ds[int(i)]

t0 = time.time()
for i in indices:
    _ = ds[int(i)]
dt = time.time() - t0
print(f'{len(indices)} reads in {dt:.2f}s — {len(indices)/dt:.0f} trials/s')
# Rough target: >=200 trials/s/worker is plenty for training I/O.

## 6. Label-form round-trip

Reconstruct the v1 agentview heatmap from v2's `contact_positions` +
`cam_agentview_*` and compare to the projection that the v1 training
pipeline uses. They should match modulo Gaussian smoothing reproducibility.

In [ ]:
from scipy.ndimage import gaussian_filter

def v2_to_agentview_heatmap(sample, sigma_px=8.0):
    pts = sample['contact_positions']
    if pts.shape[0] == 0:
        return None
    cam_pos = np.asarray(sample['cam_agentview_pos'])
    cam_mat = np.asarray(sample['cam_agentview_mat0'])
    fovy = float(sample['cam_agentview_fovy'])
    W, H = sample['cam_agentview_size']
    fy = (H / 2.0) / np.tan(np.deg2rad(fovy) / 2.0)
    fx = fy
    # MuJoCo convention -> image (negate Y row).
    R = cam_mat.T.copy()
    R[1] = -R[1]
    rel = pts - cam_pos
    pc = rel @ R.T
    z = pc[:, 2]
    valid = z > 1e-3
    u = (fx * pc[valid, 0] / z[valid] + W / 2.0).astype(int)
    v = (fy * pc[valid, 1] / z[valid] + H / 2.0).astype(int)
    keep = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    u, v = u[keep], v[keep]
    weights = np.linalg.norm(sample['contact_force_world'][valid][keep], axis=1)
    weights = weights * sample['failure_prob'] if 'failure_prob' in sample else weights
    H_map = np.zeros((H, W), dtype=np.float32)
    np.add.at(H_map, (v, u), weights)
    return gaussian_filter(H_map, sigma=sigma_px, mode='constant')

h = v2_to_agentview_heatmap(s)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].imshow(s['post_agentview_rgb']); ax[0].set_title('post_agentview_rgb'); ax[0].axis('off')
ax[1].imshow(s['post_agentview_rgb'])
if h is not None:
    ax[1].imshow(np.log1p(h), cmap='hot', alpha=0.55)
ax[1].set_title('post + v2-reconstructed log1p heatmap'); ax[1].axis('off')
plt.tight_layout(); plt.show()